[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/ml/03-data-engineering/de-acquisition.ipynb)

# Data Acquisition — APIs & Web Scraping

*AIBits Academy · Machine Learning End To End · Data Engineering*

Before you can clean, engineer, or model data, you have to get it. This is where real data-science projects actually begin.

**How to use this notebook:** run the cells top to bottom (Runtime → Run all). Each code cell is the same code you saw on the course page, so you can compare your output with the lesson. The graded exercises are at the end; try them before opening the solutions.

*Interactive animations and quiz cards stay on the course page.*

## Where Data Science Really Begins

Every earlier page in this course quietly assumed the data was already sitting in a CSV, ready to load. In the real world it almost never is. Data has to be **acquired** — pulled from a live web service, scraped off a public page, or read out of a company database — before a single line of cleaning or modelling can happen. The full arc of a real project looks like this:

> **💡 Why a Data Scientist Must Know Data Engineering**
>
> Data scientists today are not only expected to have knowledge up to "machine-learning model building" — they are expected to have at least **basic knowledge of data engineering**. The reason is simple: *if you cannot acquire data, there is no question of data cleaning, preprocessing, feature engineering, or model building.* In real industry, most of the time the data will have to be **mined by scraping or obtained using APIs** before any of the modelling you've learned so far can even begin. That is why this section is part of the course. 
>  
>  Beyond this, a new role — the **full-stack data scientist** — is emerging: someone who can also do **MLOps**, i.e. deploy the machine-learning model, analyse drift in the model, and periodically and automatically re-train it (the machine-learning engineer's role). An interested learner who wants to become a true end-to-end full-stack data scientist can look to our separate **MLOps course** for that deployment-and-monitoring half of the diagram above.

## Method 1 — APIs (Application Programming Interfaces)

An **API** is a structured "front door" a service opens so that programs — not humans clicking a browser — can request data and get back a clean, machine-readable response (almost always **JSON**). When a provider offers an API, it is nearly always the best acquisition method: the data is structured, reliable, and you're using the service the way it was designed to be used. The trade-off is that APIs are usually rate-limited and often require an authentication key.

The Python `requests` library is the standard tool. The pattern is always the same three steps: build the URL, send a GET request, parse the JSON. Below we fetch **live** package metadata from the public PyPI JSON API (no key required) — a real, working endpoint:

In [ ]:
import requests

def get_package_info(pkg):
    url = f"https://pypi.org/pypi/{pkg}/json"   # 1. build the URL
    resp = requests.get(url, timeout=10)              # 2. send GET request
    if resp.status_code == 200:                     # 200 = success
        data = resp.json()                          # 3. parse JSON → dict
        info = data['info']
        return {
            'name':     info['name'],
            'version':  info['version'],
            'summary':  info['summary'],
            'releases': len(data['releases']),
        }
    return f"Error: status {resp.status_code}"

for pkg in ['requests', 'scikit-learn', 'pandas']:
    print(get_package_info(pkg))

Those version numbers and release counts are genuinely fetched at runtime — run the code tomorrow and they may differ, which is exactly the point of a *live* API. For services that need a key (stock prices, weather, maps), the only change is appending `&apikey=YOUR_KEY` to the URL; many providers (e.g. Alpha Vantage for stock data) issue a free key to anyone with an email address.

> **🔑 The status_code Habit**
>
> Always check `resp.status_code == 200` before trusting `resp.json()`. A `401` means bad/missing key, `429` means you've hit the rate limit, `404` means the URL is wrong. Real acquisition code is mostly error-handling — the happy path is the easy part.

## Method 2 — Web Scraping

When a website shows data publicly on a page but offers *no* API, the fallback is **web scraping**: fetch the raw HTML and extract the fields you want. `requests` downloads the page; **BeautifulSoup** parses the HTML into a searchable tree. The classic safe practice site `books.toscrape.com` exists specifically for this:

In [ ]:
import requests
from bs4 import BeautifulSoup

resp = requests.get('http://books.toscrape.com/', timeout=10)
soup = BeautifulSoup(resp.text, 'html.parser')

# Each book is an <article class="product_pod"> on the page
for book in soup.find_all('article', class_='product_pod')[:3]:
    title = book.h3.a['title']                         # attribute of a nested tag
    price = book.find('p', class_='price_color').text  # text inside a tag
    print(f"{title[:40]:40s} {price}")

The two workhorse operations are `find_all(tag, class_=...)` to get a list of matching elements, and then `.text` (inner text) or `['attr']` (attribute value) to pull the actual data out of each. When a page loads its content *dynamically with JavaScript*, plain `requests` won't see it (it only gets the initial HTML) — you then need **Selenium** to drive a real browser that runs the JavaScript first, and hand its rendered page to BeautifulSoup.

### ✓ Prefer an API when

- The provider offers one — it's structured and stable
- You need reliability and clear usage terms
- The data updates and you want clean repeat access

### ✗ Fall back to scraping only when

- No API exists but the data is publicly visible
- You've checked it's legally & ethically permitted
- You can throttle requests to not overload the site

## The Ethics & Law of Data Acquisition

Just because data is technically reachable does not mean it is yours to take. Responsible acquisition rests on three duties:

| Principle | What it means in practice |
|---|---|
| **Privacy & consent** | Collect personal data only with informed consent; anonymise where possible. |
| **Transparency** | Be clear about what you collect, how it's used, and with whom it's shared. |
| **Legal compliance** | Obey the applicable laws — GDPR (Europe), CCPA (California), India's DPDP Act, and others. |

Concrete best practices for scraping specifically: **respect `robots.txt`** (the file where a site declares which paths crawlers may not touch), apply **rate limiting** so you don't overload the server, and practise **data minimisation** — collect only what you actually need. Python's standard library can check `robots.txt` for you before you fetch anything:

In [ ]:
from urllib.robotparser import RobotFileParser

def may_i_scrape(base_url, path):
    rp = RobotFileParser()
    rp.set_url(base_url + '/robots.txt')
    rp.read()
    # '*' = rules for all bots; returns True only if the path is allowed
    return rp.can_fetch('*', base_url + path)

if may_i_scrape('http://books.toscrape.com', '/'):
    print("Allowed — proceed politely (with delays between requests).")
else:
    print("Disallowed — do not scrape this path.")

> **⚠ Real Cases That Set the Boundaries**
>
> **LinkedIn v. hiQ Labs (2017):** courts allowed scraping of *public* profile data over LinkedIn's objection — public data scraping isn't automatically illegal, but it's contested and fact-specific. **Cambridge Analytica (2018):** improperly harvested tens of millions of Facebook users' data without consent; Facebook was fined US$5 billion by the FTC — consent is not optional. **Clearview AI (since 2020):** scraped billions of web images to train facial recognition and has been fined and banned in multiple countries. The lesson across all three: *technically possible* and *legally/ethically permitted* are very different questions — always answer the second one first.

### ❓ Conceptual Q&A

---
## Graded exercises

Each exercise has a **starter cell** you complete and a **check cell** that prints ✅ or ❌. The solution is folded away underneath — try first.

In [ ]:
# --- self-check helper (used by the exercises) ---------------------------------------------
def check(name, ok):
    print(("\u2705 " if ok else "\u274c ") + name)


### Exercise 1 · Easy · Read an API response

An API returned the JSON text below. Parse it with `json.loads` and store in `versions` a dict mapping each package `name` to its `version`.

In [ ]:
import json
text = '{"packages": [{"name": "pandas", "version": "2.2.0"}, {"name": "numpy", "version": "1.26.4"}]}'
versions = None   # TODO


In [ ]:
try:
    check("two packages", versions is not None and len(versions) == 2)
    check("mapping", versions == {"pandas": "2.2.0", "numpy": "1.26.4"})
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
import json
text = '{"packages": [{"name": "pandas", "version": "2.2.0"}, {"name": "numpy", "version": "1.26.4"}]}'
versions = {p["name"]: p["version"] for p in json.loads(text)["packages"]}

```

</details>

### Exercise 2 · Medium · Scrape titles and prices from HTML

Using BeautifulSoup, extract from `html` a list `books` of `(title, price)` tuples where the price is a **float** (strip the currency symbol).

In [ ]:
from bs4 import BeautifulSoup
html = """<div><article class="pod"><h3><a title="Deep Work">Deep...</a></h3><p class="price">£45.50</p></article>
<article class="pod"><h3><a title="Atomic Habits">Atomic...</a></h3><p class="price">£12.00</p></article></div>"""
books = None   # TODO


In [ ]:
try:
    check("two books", books is not None and len(books) == 2)
    check("parsed values", books == [("Deep Work", 45.5), ("Atomic Habits", 12.0)])
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
from bs4 import BeautifulSoup
html = """<div><article class="pod"><h3><a title="Deep Work">Deep...</a></h3><p class="price">£45.50</p></article>
<article class="pod"><h3><a title="Atomic Habits">Atomic...</a></h3><p class="price">£12.00</p></article></div>"""
soup = BeautifulSoup(html, "html.parser")
books = [(a.h3.a["title"], float(a.find("p", class_="price").text.lstrip("£"))) for a in soup.find_all("article", class_="pod")]

```

</details>

### Exercise 3 · Stretch · Respect robots.txt

Feed the `rules` text to `RobotFileParser.parse` (no network needed). Store in `allowed` whether a generic bot may fetch `/products` and in `blocked` whether it may **not** fetch `/admin/users`.

In [ ]:
from urllib.robotparser import RobotFileParser
rules = "User-agent: *\nDisallow: /admin\nAllow: /products"
allowed = blocked = None   # TODO


In [ ]:
try:
    check("products are allowed", allowed is True)
    check("admin is blocked", blocked is True)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
from urllib.robotparser import RobotFileParser
rules = "User-agent: *\nDisallow: /admin\nAllow: /products"
rp = RobotFileParser()
rp.parse(rules.splitlines())
allowed = rp.can_fetch("*", "/products")
blocked = not rp.can_fetch("*", "/admin/users")

```

Scraping politely also means rate-limiting yourself and reading the site's terms; robots.txt is the minimum.

</details>

---
*Back to the course: **Machine Learning End To End → Data Acquisition — APIs & Web Scraping**.*